In [ ]:
import numpy as np
import os
import re
import matplotlib.pyplot as plt
import fabio

from scipy.stats import binned_statistic_2d

def natural_sort_key(filename):
    """Sort key that orders embedded numbers numerically (frame_2 before frame_10),
    since a plain lexicographic sort scrambles frame order for non-zero-padded filenames."""
    return [int(tok) if tok.isdigit() else tok for tok in re.split(r'(\d+)', filename)]


In [ ]:
dir_coupled = r"C:\Users\j.bantol\Documents\Data\RSM\2026-06-29_gao_sto01.3\1_EIGERfull-2Dsnapshot_RSM-STO026s_34.9-37deg_0.01deg_3s_FrameFiles\frames"
dir_rocking = r"C:\Users\j.bantol\Documents\Data\RSM\2026-06-29_gao_sto01.3\2_EIGERfull-2Dsnapshot_rocking-STO026s_34.5-36deg_0.01deg_0.1s_FrameFiles\frames"

In [ ]:
%matplotlib widget
plt.close("all")

frames = sorted([f for f in os.listdir(dir_coupled) if f.endswith((".gfrm"))], key=natural_sort_key)
print(f"Found {len(frames)} frames")

# plot reference frame
ref_obj = fabio.open(os.path.join(dir_coupled, frames[40]))
ref_data = ref_obj.data
#ref_data = np.rot90(ref_obj.data, k=1)
print("pixel size:", ref_data.shape)

fig, ax = plt.subplots(figsize=(8, 5))
img = ax.imshow(ref_data, vmax=np.percentile(ref_data, 99.5), origin="lower", cmap="viridis")
plt.colorbar(img, ax=ax, label="Intensity / counts")


ax.set_title(f"Reference frame", fontsize="12")
ax.set_xlabel(f"pixel", fontsize="11")
ax.set_ylabel(f"pixel", fontsize="11")
plt.show()

In [ ]:
# instrument parameters
wavelength = 0.71076      # Å, Mo Kα
pixel_size = 0.075        # mm (75 µm)
detector_distance = 310   # mm
center_col = 493.98       # px, beam center column
center_row = 226.5        # px, beam center row    

In [ ]:
# for each frame, convert center pixel to 2theta
def pixel_to_2theta(pixel, center_pixel, two_theta_center, detector_distance, pixel_size): 
    offset_mm = (pixel - center_pixel) * pixel_size
    delta_2theta = np.rad2deg(np.arctan(offset_mm / detector_distance))    # opposite is offset_mm
    
    return two_theta_center + delta_2theta

# convert omega, 2theta to Qy, Qz
def angles_to_Q(omega, two_theta, wavelength, chi=18.4349):     # chi=tan-1(1/6 / 1/2) for angle between (026) and (001)
    k = 2 * np.pi / wavelength
    alpha_i = np.deg2rad(omega)
    alpha_f = np.deg2rad(two_theta - omega)
    chi_r   = np.deg2rad(chi)

    # Q in diffractometer frame (before tilt)
    Q_perp = k * (np.sin(alpha_i) + np.sin(alpha_f))  # along diffractometer z
    Q_par  = k * (np.cos(alpha_i) - np.cos(alpha_f))  # along diffractometer y

    # rotate by chi to get into crystal frame (026s geometry, phi=-90)
    Qy =  Q_perp * np.sin(chi_r) + Q_par * np.cos(chi_r)
    Qz =  Q_perp * np.cos(chi_r) - Q_par * np.sin(chi_r)

    return Qy, Qz

In [ ]:
# for one frame

## scan parameters
### coupled omega-2theta scan
omega = 34.9
count_time_c = 3.0
two_theta_center = 2 * omega
chi=18.4349

In [ ]:
frames = sorted([f for f in os.listdir(dir_coupled) if f.endswith((".gfrm"))], key=natural_sort_key)
obj = fabio.open(os.path.join(dir_coupled, frames[0]))
data = obj.data.astype(float) / count_time_c

nrows = data.shape[0]
ncols = data.shape[1]

Qy_frame = []
Qz_frame = []
I_frame = []

for col in range(ncols):
    for row in range(nrows):
        intensity = data[row, col]
        if intensity <= 0:
            continue
            
        # cols to 2theta to Qy and Qz via chi rotation
        tt_pixel = pixel_to_2theta(col, center_col, two_theta_center, detector_distance, pixel_size)
        Qy_pix, Qz_pix = angles_to_Q(omega, tt_pixel, wavelength)
        
        # rows to rows with offset
        offset_row = (row - center_row) * pixel_size
        delta_2theta_row = np.rad2deg(np.arctan(offset_row / detector_distance))
        delta_Qy = (2 * np.pi / wavelength) * np.deg2rad(delta_2theta_row)
        
        Qy_frame.append(Qy_pix + delta_Qy)
        Qz_frame.append(Qz_pix)
        I_frame.append(intensity)

Qy_frame = np.array(Qy_frame)
Qz_frame = np.array(Qz_frame)
I_frame  = np.array(I_frame)

In [ ]:
I_grid, Qy_edges, Qz_edges, _ = binned_statistic_2d(Qy_frame, Qz_frame, I_frame, statistic='mean', bins=[200, 200],
                                                    range=[[Qy_frame.min(), Qy_frame.max()], [Qz_frame.min(), Qz_frame.max()]])
I_grid = np.nan_to_num(I_grid, nan=0.0)
Qy_centers = (Qy_edges[:-1] + Qy_edges[1:]) / 2
Qz_centers = (Qz_edges[:-1] + Qz_edges[1:]) / 2

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
im = ax.pcolormesh(Qy_centers, Qz_centers, np.log10(I_grid.T + 1),
                   cmap='hot',
                   vmin=np.nanpercentile(np.log10(I_grid[I_grid>0]+1), 50),
                   vmax=np.nanpercentile(np.log10(I_grid+1), 99.99))
plt.colorbar(im, ax=ax, label='log₁₀(Intensity / cps)')
ax.set_xlabel("Qy / Å⁻¹")
ax.set_ylabel("Qz / Å⁻¹")
ax.set_title(f"Single frame RSM — ω={omega:.3f}°")
plt.show()

In [ ]:
obj = fabio.open(os.path.join(dir_coupled, frames[0]))
data = obj.data

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(data.sum(axis=0))  # vs col
axes[0].set_xlabel("col")
axes[0].set_title("horizontal profile")

axes[1].plot(data.sum(axis=1))  # vs row
axes[1].set_xlabel("row")
axes[1].set_title("vertical profile")

plt.tight_layout()
plt.show()

In [ ]:
# for multiple frames

## scan parameters
### coupled omega-2theta scan
omega_start_c = 34.9
omega_step_c = 0.01
count_time_c = 3.0

### rocking scan
omega_start_r = 34.5
omega_step_r = 0.01
count_time_r = 0.1
two_theta_fixed_r = 70.1144  # fixed, 2theta = 2 * omega

In [ ]:
all_Qy = []
all_Qz = []
all_I = []
all_sctype = []

def process_data(directory, omega_start, omega_step, count_time, scan_type="coupled", two_theta_fixed=None):
    frames = sorted([f for f in os.listdir(directory) if f.endswith((".gfrm"))], key=natural_sort_key)
    print(f"For {scan_type} scan: {len(frames)} frames")
    
    for i, f in enumerate(frames):
        obj = fabio.open(os.path.join(directory, f))
        data = obj.data.astype(float) / count_time  # normalize to cps

        omega = omega_start + i * omega_step
    
        if scan_type == "coupled":
            two_theta_center = 2 * omega
        else:
            two_theta_center = two_theta_fixed
    
        # rows are perpendicular to the diffraction plane (Debye-Scherrer arc direction) -
        # they don't map onto a different in-plane Qy/Qz, so integrate them out per column
        # instead of converting row pixel position into a spurious Q shift
        intensity_per_col = data.sum(axis=0)   # shape (ncols,)
    
        # cols -> 2theta -> Qy/Qz via chi rotation, vectorized over the whole column axis
        ncols = data.shape[1]
        cols = np.arange(ncols)
        tt_pixel = pixel_to_2theta(cols, center_col, two_theta_center, detector_distance, pixel_size)
        Qy_col, Qz_col = angles_to_Q(omega, tt_pixel, wavelength)          # shape (ncols,)
    
        mask = intensity_per_col > 0
        all_Qy.append(Qy_col[mask])
        all_Qz.append(Qz_col[mask])
        all_I.append(intensity_per_col[mask])
        all_sctype.append(np.full(mask.sum(), scan_type))
    
    global Qy, Qz, I, sctype
    Qy = np.concatenate(all_Qy)
    Qz = np.concatenate(all_Qz)
    I = np.concatenate(all_I)
    sctype = np.concatenate(all_sctype)


In [ ]:
# process a scan
process_data(dir_coupled, omega_start_c, omega_step_c, count_time_c, scan_type="coupled")
process_data(dir_rocking, omega_start_r, omega_step_r, count_time_r, scan_type="rocking", two_theta_fixed=two_theta_fixed_r)


In [ ]:
print(f"Total points: {len(Qy)}")
print(f"Qy range: {Qy.min():.4f} to {Qy.max():.4f} Å⁻¹")
print(f"Qz range: {Qz.min():.4f} to {Qz.max():.4f} Å⁻¹")

In [ ]:
grids = {}
for label in ['coupled', 'rocking']:
    mask = sctype == label
    Qy_s, Qz_s, I_s = Qy[mask], Qz[mask], I[mask]

    I_grid, Qy_edges, Qz_edges, _ = binned_statistic_2d(
        Qy_s, Qz_s, I_s, statistic='mean', bins=[200, 200],
        range=[[Qy_s.min(), Qy_s.max()], [Qz_s.min(), Qz_s.max()]]
    )
    I_grid = np.nan_to_num(I_grid, nan=0.0)
    Qy_centers = (Qy_edges[:-1] + Qy_edges[1:]) / 2
    Qz_centers = (Qz_edges[:-1] + Qz_edges[1:]) / 2

    grids[label] = (I_grid, Qy_centers, Qz_centers)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, label in zip(axes, ['coupled', 'rocking']):
    I_grid, Qy_centers, Qz_centers = grids[label]
    im = ax.pcolormesh(Qy_centers, Qz_centers, np.log10(I_grid.T + 1),
                        cmap='hot',
                        vmin=np.nanpercentile(np.log10(I_grid[I_grid > 0] + 1), 50),
                        vmax=np.nanpercentile(np.log10(I_grid + 1), 99.99))
    plt.colorbar(im, ax=ax, label='log₁₀(Intensity / cps)')
    ax.set_xlabel("Qy / Å⁻¹")
    ax.set_ylabel("Qz / Å⁻¹")
    ax.set_title(f"{label} scan RSM ({(sctype == label).sum()} pts)")

plt.tight_layout()
plt.show()


In [ ]:
for label, (I_grid, _, _) in grids.items():
    print(f"{label}: valid intensity sum = {np.nansum(I_grid):.1f}, NaN count = {np.sum(np.isnan(I_grid))}")
print("Qy sample:", Qy[:5])
print("Qz sample:", Qz[:5])
print("I sample:", I[:5])


In [ ]:
# grid and plot
I_grid, Qy_edges, Qz_edges, _ = binned_statistic_2d(Qy, Qz, I, statistic="mean", bins=[600, 600], range=[[Qy.min(), Qy.max()], [Qz.min(), Qz.max()]])

#Qy_centers = (Qy_edges[:-1] + Qy_edges[1:]) / 2
#Qz_centers = (Qz_edges[:-1] + Qz_edges[1:]) / 2

I_grid = np.nan_to_num(I_grid, nan=0.0)  # replace NaN with 0 for plotting

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, label, color in zip(axes, ['rocking', 'coupled'], ['C0', 'C1']):
    mask = sctype == label
    ax.scatter(Qy[mask], Qz[mask], c=np.log10(I[mask]+1), s=0.1,
               cmap='viridis')
    ax.set_xlabel("Qy / Å⁻¹")
    ax.set_ylabel("Qz / Å⁻¹")
    ax.set_title(f"{label}: {mask.sum()} points")

plt.tight_layout()
plt.show()


In [ ]:
from scipy.ndimage import maximum_filter

def find_local_peaks(I_grid, Qy_centers, Qz_centers, min_distance=5, top_n=2):
    """Return the top_n distinct local intensity maxima (film/substrate),
    instead of a single global argmax which can pick a different physical
    peak in each scan if their relative brightness differs.
    I_grid axis 0 is Qy, axis 1 is Qz (binned_statistic_2d(Qy, Qz, ...) convention)."""
    footprint = np.ones((min_distance, min_distance))
    local_max = (I_grid == maximum_filter(I_grid, footprint=footprint)) & (I_grid > 0)
    iQy_all, iQz_all = np.nonzero(local_max)
    order = np.argsort(I_grid[iQy_all, iQz_all])[::-1]

    peaks = []
    for idx in order:
        iQy, iQz = iQy_all[idx], iQz_all[idx]
        if any(abs(iQy - pQy) < min_distance and abs(iQz - pQz) < min_distance for pQy, pQz, _ in peaks):
            continue
        peaks.append((iQy, iQz, I_grid[iQy, iQz]))
        if len(peaks) == top_n:
            break

    return [(Qy_centers[iQy], Qz_centers[iQz], intensity) for iQy, iQz, intensity in peaks]

for label in ['coupled', 'rocking']:
    I_grid, Qy_centers, Qz_centers = grids[label]
    peaks = find_local_peaks(I_grid, Qy_centers, Qz_centers)
    print(f"{label} scan:")
    for Qy_pk, Qz_pk, intensity in peaks:
        print(f"  Qy={Qy_pk:.4f}, Qz={Qz_pk:.4f}  (I={intensity:.1f} cps)")


In [ ]:
def peak_near(I_grid, Qy_centers, Qz_centers, Qy_target, Qz_target, window=0.1,
              exclude_Qy=None, exclude_Qz=None, exclude_radius=0.0):
    """Max intensity within a Qy/Qz window around a target location, instead of
    an unconstrained global search that can latch onto an unrelated bright
    feature (background streak, noise) in a noisier/weaker-signal scan.
    Optionally zero out a disk around another peak (e.g. the substrate) first,
    since substrate and film can sit closer together than the window radius."""
    iQy_mask = np.abs(Qy_centers - Qy_target) < window
    iQz_mask = np.abs(Qz_centers - Qz_target) < window
    Qy_sub = Qy_centers[iQy_mask]
    Qz_sub = Qz_centers[iQz_mask]
    sub = I_grid[np.ix_(iQy_mask, iQz_mask)].copy()

    if exclude_Qy is not None:
        Yg, Zg = np.meshgrid(Qy_sub, Qz_sub, indexing='ij')
        sub[(Yg - exclude_Qy)**2 + (Zg - exclude_Qz)**2 < exclude_radius**2] = 0

    if sub.size == 0 or np.all(sub == 0):
        return None
    iQy_local, iQz_local = np.unravel_index(np.argmax(sub), sub.shape)
    return Qy_sub[iQy_local], Qz_sub[iQz_local], sub[iQy_local, iQz_local]

# use the coupled scan's clearly-resolved peaks (from find_local_peaks above) as reference
# locations, and exclude the substrate peak itself so it doesn't dominate the window
substrate_ref_Qy, substrate_ref_Qz = 3.2151, 9.6349
film_ref_Qy, film_ref_Qz = 3.2245, 9.6869

for label in ['coupled', 'rocking']:
    I_grid, Qy_centers, Qz_centers = grids[label]
    result = peak_near(I_grid, Qy_centers, Qz_centers, film_ref_Qy, film_ref_Qz, window=0.1,
                        exclude_Qy=substrate_ref_Qy, exclude_Qz=substrate_ref_Qz, exclude_radius=0.025)
    print(f"{label} scan, near expected film location (substrate excluded): {result}")


In [ ]:
# zoom into the substrate/film cluster with a SHARED color scale across both scans,
# so brightness is directly comparable (the earlier plots each used their own
# per-panel percentile range, which can visually mislead a side-by-side comparison)
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
vmin_shared, vmax_shared = 2.0, 3.6   # log10(cps), covers up to ~4000 cps

for ax, label in zip(axes, ['coupled', 'rocking']):
    I_grid, Qy_centers, Qz_centers = grids[label]
    im = ax.pcolormesh(Qy_centers, Qz_centers, np.log10(I_grid.T + 1),
                        cmap='hot', vmin=vmin_shared, vmax=vmax_shared)
    plt.colorbar(im, ax=ax, label='log₁₀(Intensity / cps)')
    ax.set_xlim(3.05, 3.35)
    ax.set_ylim(9.55, 9.75)
    ax.set_xlabel("Qy / Å⁻¹")
    ax.set_ylabel("Qz / Å⁻¹")
    ax.set_title(f"{label} scan (zoomed, shared color scale)")

plt.tight_layout()
plt.show()


In [ ]:
from scipy.interpolate import RegularGridInterpolator

def rod_profile_from_point(I_grid, Qy_centers, Qz_centers, origin, direction, max_dist=0.15, n=300):
    """Interpolated intensity walking from `origin` along `direction` in (Qy, Qz).
    Anchoring at a confirmed peak location (rather than an eyeballed line through
    the plot) matters here: the substrate->film offset is (0.0094, 0.052) - almost
    straight up in Qz - which is a much steeper direction than the diagonal streak
    looks like by eye, so a guessed line can pass right by the actual peaks while
    still crossing other, broader parts of the streak nearby."""
    direction = np.asarray(direction, dtype=float)
    direction = direction / np.linalg.norm(direction)
    dist = np.linspace(0, max_dist, n)
    Qy_line = origin[0] + dist * direction[0]
    Qz_line = origin[1] + dist * direction[1]
    interp = RegularGridInterpolator((Qy_centers, Qz_centers), I_grid, bounds_error=False, fill_value=0)
    I_line = interp(np.column_stack([Qy_line, Qz_line]))
    return dist, I_line

# direction confirmed from the coupled scan's clearly-resolved substrate -> film pair
# (from the very first find_local_peaks result: substrate (3.2151, 9.6349), film (3.2245, 9.6869))
direction = np.array([3.2245, 9.6869]) - np.array([3.2151, 9.6349])
film_offset_dist = np.linalg.norm(direction)

# each scan is anchored at its OWN substrate peak location (these already agree to
# within one grid bin between scans), walking in that same confirmed direction
substrate_by_scan = {'coupled': (3.2151, 9.6349), 'rocking': (3.2097, 9.6352)}

fig, ax = plt.subplots(figsize=(9, 5))
for label, origin in substrate_by_scan.items():
    I_grid, Qy_centers, Qz_centers = grids[label]
    dist, I_line = rod_profile_from_point(I_grid, Qy_centers, Qz_centers, origin, direction, max_dist=0.15)
    ax.plot(dist, np.clip(I_line, 1e-3, None), label=label, marker='.')

ax.axvline(film_offset_dist, color='gray', linestyle='--', label='coupled-scan film offset')
ax.set_yscale('log')
ax.set_xlabel("distance from substrate peak toward film direction  (\u00c5\u207b\u00b9)")
ax.set_ylabel("Intensity / cps")
ax.legend()
ax.set_title("Intensity from substrate peak toward the confirmed film direction")
fig.subplots_adjust(bottom=0.12, left=0.1, right=0.97, top=0.93)
plt.show()


In [ ]:
# overlay the substrate location (from find_local_peaks) and the PREDICTED film
# location (substrate + the coupled-scan-confirmed offset) directly on the actual
# zoomed maps, so we can see exactly where the prediction falls relative to
# whatever is visually there - rather than trusting only a 1D line cut through it
substrate_by_scan = {'coupled': (3.2151, 9.6349), 'rocking': (3.2097, 9.6352)}
film_offset = np.array([3.2245, 9.6869]) - np.array([3.2151, 9.6349])

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, label in zip(axes, ['coupled', 'rocking']):
    I_grid, Qy_centers, Qz_centers = grids[label]
    im = ax.pcolormesh(Qy_centers, Qz_centers, np.log10(I_grid.T + 1),
                        cmap='hot', vmin=2.0, vmax=3.6)
    plt.colorbar(im, ax=ax, label='log₁₀(Intensity / cps)')

    sub_Qy, sub_Qz = substrate_by_scan[label]
    film_Qy, film_Qz = sub_Qy + film_offset[0], sub_Qz + film_offset[1]

    ax.plot(sub_Qy, sub_Qz, marker='+', color='cyan', markersize=18, markeredgewidth=2, label='substrate (measured)')
    ax.plot(film_Qy, film_Qz, marker='x', color='lime', markersize=18, markeredgewidth=2, label='predicted film location')

    ax.set_xlim(3.05, 3.35)
    ax.set_ylim(9.55, 9.75)
    ax.set_xlabel("Qy / \u00c5\u207b\u00b9")
    ax.set_ylabel("Qz / \u00c5\u207b\u00b9")
    ax.set_title(f"{label} scan (zoomed)")
    ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()
